## Lost all MLFlow runs so used Gemini to find them again

In [ ]:
"""
Recovers results from mlflow runs whose tracking-store metadata was lost,
but whose model_report.json artifacts survived on disk.

Run this from your project root (msc-final-project/), with the same
venv you used for training.

What it does:
1. Reads models/completed_runs.txt to get the ordered list of run_names
   (each line = one completed run, in completion order).
2. Reads every mlruns/0/<run_id>/artifacts/model_report.json, sorted by
   file modification time (same chronological order as completed_runs.txt).
3. Zips the two together by order to reconstruct config -> metrics.
4. Saves everything to recovered_results.csv.
5. Optionally re-logs each as a real mlflow run (metrics + run_name tag only
   - NOT the original hyperparameters, which weren't saved anywhere on disk).

IMPORTANT CAVEAT:
The actual tuned XGBoost hyperparameters (best_params) for each run were
only ever sent via mlflow.log_params(), which failed silently along with
the rest of the tracking-store writes. They are NOT recoverable from disk.
This script only recovers metrics + the config encoded in run_name
(data sources, k, n_splits, pca, event_col, conflict_only).
"""

import json
from pathlib import Path

import pandas as pd

RUNS_DIR = Path("mlruns/0")
COMPLETED_RUNS_FILE = Path("models/completed_runs.txt")
OUTPUT_CSV = Path("recovered_results.csv")


def load_run_names() -> list[str]:
    if not COMPLETED_RUNS_FILE.exists():
        print(
            f"WARNING: {COMPLETED_RUNS_FILE} not found. run_name column will be empty."
        )
        return []
    with open(COMPLETED_RUNS_FILE) as f:
        return [line.strip() for line in f if line.strip()]


def load_result_folders() -> list[Path]:
    folders = [
        d
        for d in RUNS_DIR.iterdir()
        if d.is_dir() and (d / "artifacts" / "model_report.json").exists()
    ]
    # Sort chronologically to match completed_runs.txt's append order
    folders.sort(key=lambda d: (d / "artifacts" / "model_report.json").stat().st_mtime)
    return folders


def main():
    run_names = load_run_names()
    folders = load_result_folders()

    print(f"Found {len(folders)} run folders with model_report.json")
    print(f"Found {len(run_names)} completed run names")

    if len(folders) != len(run_names):
        print(
            "WARNING: counts don't match — the chronological zip below may be "
            "misaligned. Spot-check a few rows before trusting run_name mapping."
        )

    records = []
    for i, folder in enumerate(folders):
        report_path = folder / "artifacts" / "model_report.json"
        with open(report_path) as f:
            results = json.load(f)

        run_name = run_names[i] if i < len(run_names) else None
        records.append(
            {
                "run_folder": folder.name,
                "run_name": run_name,
                **results,
            }
        )

    df = pd.DataFrame(records)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved {len(df)} rows to {OUTPUT_CSV}")
    print(df.head())

    return df


if __name__ == "__main__":
    df = main()

In [8]:
df

,run_folder,run_name,optimal_threshold,n_predictors,onset_aupr,onset_precision_class1,onset_recall_class1,onset_f1_class1,active_aupr,active_precision_class1,active_recall_class1,active_f1_class1
0,3caf6941b0a54e48a96f6ac37188ef3f,acled_sub_food_rain_text_conflict_pca_0.25_4,0.3174,57,0.4277,0.4213,1.0000,0.5928,0.4177,0.4259,1.0000,0.5974
1,1b5093fa358145c09c70b94a4bd08413,acled_sub_food_rain_text_conflict_0.25_4,0.4213,801,0.5101,0.4213,1.0000,0.5928,0.4180,0.4259,1.0000,0.5974
2,855e0ce64a804171944499c8138e87d4,acled_sub_food_rain_text_conflict_pca_0.25_5,0.1409,57,0.3908,0.4292,1.0000,0.6007,0.3997,0.4396,0.9891,0.6087
3,77c398bfb8d74293a0e600e199c29402,acled_sub_food_rain_text_conflict_0.25_5,0.3445,801,0.4271,0.4213,1.0000,0.5928,0.4109,0.4259,1.0000,0.5974
4,de95ae38455f45ada5a8d9cd2a9cefde,acled_sub_food_rain_text_all_pca_0.25_4,0.1588,76,0.4550,0.4213,1.0000,0.5928,0.4485,0.4259,1.0000,0.5974
...,...,...,...,...,...,...,...,...,...,...,...,...
315,a53ae4e3a93543e99f102ffd4704fa7f,acled_event_0.75_5,0.0521,10,0.3729,0.3472,1.0000,0.5155,0.3777,0.3163,1.0000,0.4806
316,61eccc32fb844c9ba8789f3d41e3313f,acled_sub_1_4,0.4432,28,0.3595,0.3364,0.5455,0.4162,0.3737,0.3571,0.6034,0.4487
317,d071ad321b4f44b7b424b28273a7c0b1,acled_sub_1_5,0.4171,28,0.3565,0.3196,0.4697,0.3804,0.3498,0.3434,0.5862,0.4331
318,cadfcfe463bb4386b92f94ac0efc10ae,acled_event_1_4,0.4825,10,0.3500,0.3235,0.5000,0.3929,0.4499,0.3678,0.5517,0.4414


In [ ]:
"""
Adds structured config columns to recovered_results.csv, parsed from run_name.

run_name format (from run_models.ipynb):
    f"{which_data}{pca_str}_{k}_{n}"
where:
    which_data = f"acled_{event_str}{food_str}{rain_str}{text_str}"
    event_str  = "event" | "sub"
    food_str   = "_food" | ""
    rain_str   = "_rain" | ""
    text_str   = "_text_conflict" | "_text_all" | ""
    pca_str    = "_pca" | ""

k and n are always the last two underscore-separated tokens, so we split
on that rather than assuming exact positions for the optional pieces -
robust even if some run_names are missing a segment.

Run from your project root: python parse_run_name.py
"""

from pathlib import Path

INPUT_CSV = Path("recovered_results.csv")
OUTPUT_CSV = Path("runs.csv")  # overwrite in place


def parse_run_name(name: str) -> dict:
    if not isinstance(name, str) or not name:
        return {
            "food": None,
            "rain": None,
            "text": None,
            "conflict_text": None,
            "pca": None,
            "k": None,
            "n": None,
        }

    parts = name.split("_")

    return {
        "food": "_food" in name,
        "rain": "_rain" in name,
        "text": "_text_" in name,
        "conflict_text": "_text_conflict" in name,
        "pca": "_pca" in name,
        "k": float(parts[-2]),
        "n": int(parts[-1]),
    }


def main():
    df = pd.read_csv(INPUT_CSV)

    parsed = df["run_name"].apply(parse_run_name).apply(pd.Series)
    df = pd.concat([df, parsed], axis=1)

    # sanity check: any rows that failed to parse (bad run_name / off-by-one from earlier zip)
    bad_rows = df[df["k"].isna()]
    if len(bad_rows):
        print(f"WARNING: {len(bad_rows)} rows failed to parse run_name — check these:")
        print(bad_rows[["run_folder", "run_name"]])

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved {len(df)} rows with parsed columns to {OUTPUT_CSV}")
    print(
        df[["run_name", "food", "rain", "text", "conflict_text", "pca", "k", "n"]].head(
            10
        )
    )


if __name__ == "__main__":
    runs = main()

ValueError: cannot reindex on an axis with duplicate labels

In [11]:
runs = pd.read_csv("recovered_results.csv")

In [12]:
runs

,run_folder,run_name,optimal_threshold,n_predictors,onset_aupr,onset_precision_class1,onset_recall_class1,onset_f1_class1,active_aupr,active_precision_class1,active_recall_class1,active_f1_class1,food,rain,text,conflict_text,pca,k,n
0,3caf6941b0a54e48a96f6ac37188ef3f,acled_sub_food_rain_text_conflict_pca_0.25_4,0.3174,57,0.4277,0.4213,1.0000,0.5928,0.4177,0.4259,1.0000,0.5974,True,True,True,True,True,0.25,4
1,1b5093fa358145c09c70b94a4bd08413,acled_sub_food_rain_text_conflict_0.25_4,0.4213,801,0.5101,0.4213,1.0000,0.5928,0.4180,0.4259,1.0000,0.5974,True,True,True,True,False,0.25,4
2,855e0ce64a804171944499c8138e87d4,acled_sub_food_rain_text_conflict_pca_0.25_5,0.1409,57,0.3908,0.4292,1.0000,0.6007,0.3997,0.4396,0.9891,0.6087,True,True,True,True,True,0.25,5
3,77c398bfb8d74293a0e600e199c29402,acled_sub_food_rain_text_conflict_0.25_5,0.3445,801,0.4271,0.4213,1.0000,0.5928,0.4109,0.4259,1.0000,0.5974,True,True,True,True,False,0.25,5
4,de95ae38455f45ada5a8d9cd2a9cefde,acled_sub_food_rain_text_all_pca_0.25_4,0.1588,76,0.4550,0.4213,1.0000,0.5928,0.4485,0.4259,1.0000,0.5974,True,True,True,False,True,0.25,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,a53ae4e3a93543e99f102ffd4704fa7f,acled_event_0.75_5,0.0521,10,0.3729,0.3472,1.0000,0.5155,0.3777,0.3163,1.0000,0.4806,False,False,False,False,False,0.75,5
316,61eccc32fb844c9ba8789f3d41e3313f,acled_sub_1_4,0.4432,28,0.3595,0.3364,0.5455,0.4162,0.3737,0.3571,0.6034,0.4487,False,False,False,False,False,1.00,4
317,d071ad321b4f44b7b424b28273a7c0b1,acled_sub_1_5,0.4171,28,0.3565,0.3196,0.4697,0.3804,0.3498,0.3434,0.5862,0.4331,False,False,False,False,False,1.00,5
318,cadfcfe463bb4386b92f94ac0efc10ae,acled_event_1_4,0.4825,10,0.3500,0.3235,0.5000,0.3929,0.4499,0.3678,0.5517,0.4414,False,False,False,False,False,1.00,4
